# 🍲 NOSH Recipe Dataset: Analysis & Data Cleaning Pipeline
This notebook performs an in-depth data quality analysis and end-to-end cleaning pipeline on the raw recipe datasets (`data_recipes/*.json`).

### Objectives:
1. **Load and Consolidate**: Aggregate raw recipes from AllRecipes, Epicurious, and Food Network.
2. **Comprehensive EDA**: Identify missing values, corrupted records, empty objects, duplicate recipes, HTML artifacts, and advertisement noise.
3. **Data Cleaning Pipeline**:
   - Strip extensive advertisement noise (e.g., `ADVERTISEMENT` strings in ingredients).
   - Clean and normalize titles, ingredients, and instructions (HTML decoding, whitespace trimming).
   - Filter out empty, placeholder, or unusable recipes.
   - Smart deduplication prioritizing entries with picture links.
4. **Feature Engineering**: Extract ingredient counts, instruction lengths, step counts, and recipe sources.
5. **Quality Verification & Export**: Validate post-cleaning data integrity and export the clean dataset for downstream RAG ingestion.


In [1]:
# -----------------------------------------------------------------------------
# STEP 1: Ingestion & Consolidating Raw Recipe Datasets
# -----------------------------------------------------------------------------
# We load all raw JSON files from the data_recipes directory and combine them into
# a single unified pandas DataFrame for downstream analysis and cleaning.

import os
import glob
import json
import re
import html
import pandas as pd

# List and load each source file
raw_frames = []
for file_path in sorted(glob.glob("data_recipes/*.json")):
    # Skip any previously generated clean files
    if "cleaned" in file_path:
        continue
    
    with open(file_path, "r", encoding="utf-8") as f:
        raw_dict = json.load(f)
        
    # Convert dictionary with recipe IDs as keys into a DataFrame
    frame = pd.DataFrame.from_dict(raw_dict, orient="index")
    frame.index.name = "id"
    frame = frame.reset_index()
    frame["source_file"] = file_path
    
    # Identify source website from file name
    if "ar" in file_path:
        frame["source"] = "AllRecipes"
    elif "epi" in file_path:
        frame["source"] = "Epicurious"
    elif "fn" in file_path:
        frame["source"] = "FoodNetwork"
    else:
        frame["source"] = "Unknown"
        
    raw_frames.append(frame)
    print(f"Loaded '{file_path}' -> {len(frame):,} recipes")

# Combine all frames into df_raw
df_raw = pd.concat(raw_frames, ignore_index=True)
df_raw["id"] = df_raw["id"].astype(str)

print("-" * 60)
print(f"Total raw dataset shape: {df_raw.shape}")
print(f"Total memory usage: {df_raw.memory_usage().sum() / (1024 * 1024):.2f} MB")
df_raw.head(3)


Loaded 'data_recipes\recipes_raw_nosource_ar.json' -> 39,802 recipes


Loaded 'data_recipes\recipes_raw_nosource_epi.json' -> 25,323 recipes


Loaded 'data_recipes\recipes_raw_nosource_fn.json' -> 60,039 recipes
------------------------------------------------------------
Total raw dataset shape: (125164, 7)
Total memory usage: 6.68 MB


,id,title,ingredients,instructions,picture_link,source_file,source
0,rmK12Uau.ntP510KeImX506H6Mr6jTu,Slow Cooker Chicken and Dumplings,"[4 skinless, boneless chicken breast halves AD...","Place the chicken, butter, soup, and onion in ...",55lznCYBbs2mT8BTx6BTkLhynGHzM.S,data_recipes\recipes_raw_nosource_ar.json,AllRecipes
1,5ZpZE8hSVdPk2ZXo1mZTyoPWJRSCPSm,Awesome Slow Cooker Pot Roast,[2 (10.75 ounce) cans condensed cream of mushr...,"In a slow cooker, mix cream of mushroom soup, ...",QyrvGdGNMBA2lDdciY0FjKu.77MM0Oe,data_recipes\recipes_raw_nosource_ar.json,AllRecipes
2,clyYQv.CplpwJtjNaFGhx0VilNYqRxu,Brown Sugar Meatloaf,"[1/2 cup packed brown sugar ADVERTISEMENT, 1/2...",Preheat oven to 350 degrees F (175 degrees C)....,LVW1DI0vtlCrpAhNSEQysE9i/7rJG56,data_recipes\recipes_raw_nosource_ar.json,AllRecipes


## 2. Exploratory Data Analysis & Identifying Quality Issues
In this section, we analyze the raw dataset to uncover corrupted records, missing fields, noise artifacts, and duplicate entries.


In [2]:
# -----------------------------------------------------------------------------
# STEP 2.1: Missing Values & Null Breakdown Across Sources
# -----------------------------------------------------------------------------
# Let's inspect missing values across all columns and check how nulls are distributed
# across the individual source datasets.

# Overall null analysis
null_summary = pd.DataFrame({
    "Total Rows": len(df_raw),
    "Null Count": df_raw.isna().sum(),
    "Null %": (df_raw.isna().mean() * 100).round(2)
})
print("=== Overall Missing Values ===")
print(null_summary)

# Source-wise null breakdown
print("\n=== Missing Values Broken Down by Source ===")
source_nulls = df_raw.groupby("source").agg(
    total_recipes=("id", "count"),
    null_title=("title", lambda s: s.isna().sum()),
    null_ingredients=("ingredients", lambda s: s.isna().sum()),
    null_instructions=("instructions", lambda s: s.isna().sum()),
    null_pictures=("picture_link", lambda s: s.isna().sum())
).reset_index()
print(source_nulls.to_string(index=False))

# Identify completely empty recipe entries (where all core fields are null)
completely_empty = df_raw[
    df_raw["title"].isna() & 
    df_raw["ingredients"].isna() & 
    df_raw["instructions"].isna()
]
print(f"\nCompletely empty entries (all core fields null): {len(completely_empty):,}")


=== Overall Missing Values ===
              Total Rows  Null Count  Null %
id                125164           0    0.00
title             125164         569    0.45
ingredients       125164         517    0.41
instructions      125164         691    0.55
picture_link      125164       42571   34.01
source_file       125164           0    0.00
source            125164           0    0.00

=== Missing Values Broken Down by Source ===
     source  total_recipes  null_title  null_ingredients  null_instructions  null_pictures
 AllRecipes          39802         280               280                280            280
 Epicurious          25323           0                 0                  0          13204
FoodNetwork          60039         289               237                411          29087

Completely empty entries (all core fields null): 517


In [3]:
# -----------------------------------------------------------------------------
# STEP 2.2: Deep Dive into Ingredients: Advertisements & Empty Lists
# -----------------------------------------------------------------------------
# Scraping artifacts are very prevalent in the ingredients column:
# 1. 'ADVERTISEMENT' appears hundreds of thousands of times (both standalone and appended).
# 2. Some recipes have empty lists: [].
# 3. HTML tags like <em> and <sup> are sometimes present.

# 1. Check ingredients type distribution and empty lists
ing_not_na = df_raw["ingredients"].dropna()
empty_list_count = df_raw["ingredients"].apply(lambda x: isinstance(x, list) and len(x) == 0).sum()
print(f"Recipes with null ingredients: {df_raw['ingredients'].isna().sum():,}")
print(f"Recipes with empty list []: {empty_list_count:,}")

# 2. Count ADVERTISEMENT occurrences
standalone_ad_count = 0
appended_ad_count = 0
sample_ads = []

for ing_list in ing_not_na:
    if isinstance(ing_list, list):
        for item in ing_list:
            if not isinstance(item, str):
                continue
            trimmed = item.strip()
            if trimmed.upper() == "ADVERTISEMENT":
                standalone_ad_count += 1
            elif "ADVERTISEMENT" in trimmed.upper():
                appended_ad_count += 1
                if len(sample_ads) < 5:
                    sample_ads.append(trimmed)

print(f"\nStandalone 'ADVERTISEMENT' items: {standalone_ad_count:,}")
print(f"Items with appended 'ADVERTISEMENT': {appended_ad_count:,}")
print(f"Total advertisement contaminated items: {standalone_ad_count + appended_ad_count:,}")

print("\nSamples of dirty ingredient items:")
for s in sample_ads:
    print(f"  • {s}")


Recipes with null ingredients: 517
Recipes with empty list []: 1,676



Standalone 'ADVERTISEMENT' items: 39,519
Items with appended 'ADVERTISEMENT': 361,292
Total advertisement contaminated items: 400,811

Samples of dirty ingredient items:
  • 4 skinless, boneless chicken breast halves ADVERTISEMENT
  • 2 tablespoons butter ADVERTISEMENT
  • 2 (10.75 ounce) cans condensed cream of chicken soup ADVERTISEMENT
  • 1 onion, finely diced ADVERTISEMENT
  • 2 (10 ounce) packages refrigerated biscuit dough, torn into pieces ADVERTISEMENT


In [4]:
# -----------------------------------------------------------------------------
# STEP 2.3: Deep Dive into Instructions: Blank Text, Formatting & Junk
# -----------------------------------------------------------------------------
# We examine issues in instructions:
# - Empty or whitespace-only strings.
# - Multi-line strings with inconsistent line breaks (\r\n vs \n).
# - Placeholder or corrupted instructions (e.g., single characters like ';' or '.').

inst_series = df_raw["instructions"].fillna("")
empty_inst_count = (inst_series.str.strip() == "").sum()
short_inst_count = ((inst_series.str.strip() != "") & (inst_series.str.strip().str.len() < 10)).sum()

print(f"Recipes with empty/whitespace-only instructions: {empty_inst_count:,}")
print(f"Recipes with suspiciously short instructions (<10 chars): {short_inst_count:,}")

# Inspect some suspicious short instructions
sample_short = df_raw[
    (df_raw["instructions"].notna()) & 
    (df_raw["instructions"].str.strip().str.len() < 10) &
    (df_raw["instructions"].str.strip().str.len() > 0)
][["title", "instructions", "source"]].head(5)

print("\nSample recipes with invalid/corrupt instructions:")
print(sample_short.to_string(index=False))


Recipes with empty/whitespace-only instructions: 714
Recipes with suspiciously short instructions (<10 chars): 20



Sample recipes with invalid/corrupt instructions:
                    title instructions      source
                    Title            ; FoodNetwork
        Southern Stuffing           .; FoodNetwork
                    Title            ; FoodNetwork
          Gas Station Cue            ; FoodNetwork
Dark Chocolate as a Snack            ; FoodNetwork


In [5]:
# -----------------------------------------------------------------------------
# STEP 2.4: Deep Dive into Titles & Duplicates
# -----------------------------------------------------------------------------
# 1. Check for placeholder titles like 'Title', 'Untitled', 'Recipe'.
# 2. Check for duplicate IDs and duplicate recipes across datasets.

# Check bogus titles
bogus_titles = df_raw[df_raw["title"].astype(str).str.strip().str.lower().isin(["title", "untitled", "test", "recipe", ""])]
print(f"Recipes with bogus placeholder titles: {len(bogus_titles)}")

# Check duplicate IDs
dup_ids = df_raw["id"].duplicated().sum()
print(f"Duplicate IDs across files: {dup_ids}")

# Check duplicate recipe content (same title and instructions)
dup_recipes = df_raw.duplicated(subset=["title", "instructions"]).sum()
print(f"Duplicate recipes (identical title + instructions): {dup_recipes:,}")

# Sample duplicate recipes across different sources
dup_mask = df_raw["title"].notna() & df_raw["instructions"].notna() & df_raw.duplicated(subset=["title", "instructions"], keep=False)
print(f"Total rows involved in duplicate sets: {dup_mask.sum():,}")


Recipes with bogus placeholder titles: 6
Duplicate IDs across files: 0


Duplicate recipes (identical title + instructions): 1,856
Total rows involved in duplicate sets: 2,424


## 3. Data Cleaning Pipeline
Now that we have surveyed and quantified all data anomalies, we implement a modular, robust cleaning pipeline:
1. **Title Cleaning**: HTML decoding, tag removal, whitespace standardization, and filtering out placeholder titles.
2. **Ingredient Cleaning**: Removing `ADVERTISEMENT` strings, HTML stripping, unescaping entities, trimming whitespace, and filtering out empty/junk elements.
3. **Instruction Cleaning**: Unescaping HTML, translating `<br>` to clean line breaks, stripping residual tags, normalizing line endings, and trimming whitespace.
4. **Validation & Filtering**: Dropping corrupted recipes that lack title, ingredients, or valid instructions.
5. **Deduplication**: Removing redundant duplicates while prioritizing records that have valid picture links.


In [6]:
# -----------------------------------------------------------------------------
# STEP 3.1: Defining Modular Cleaning Functions
# -----------------------------------------------------------------------------

# Pre-compiled regex patterns for maximum efficiency across 125k+ records
AD_PATTERN = re.compile(r'\bADVERTISEMENT\b', re.IGNORECASE)
HTML_TAG_PATTERN = re.compile(r'<[^>]+>')
WHITESPACE_PATTERN = re.compile(r'\s+')
BOGUS_TITLES = {"title", "untitled", "test", "recipe", "", "none", "n/a"}
JUNK_INGREDIENTS = {"", "advertisement", "em", "/em", "sup", "/sup", "n/a", "none", ";", ".", ""}

def clean_title(title: str) -> str | None:
    """
    Cleans and standardizes recipe titles.
    - Decodes HTML entities (e.g. &amp; -> &)
    - Strips HTML tags if present
    - Collapses redundant whitespace
    - Returns None if empty or matches placeholder junk (e.g. 'Title')
    """
    if pd.isna(title) or not isinstance(title, str):
        return None
    t = html.unescape(title)
    t = HTML_TAG_PATTERN.sub('', t)
    t = WHITESPACE_PATTERN.sub(' ', t).strip()
    if t.lower() in BOGUS_TITLES or len(t) < 2:
        return None
    return t

def clean_ingredients(raw_ingredients: list) -> list[str]:
    """
    Cleans a list of ingredients for a recipe.
    - Verifies list type
    - Removes standalone and appended 'ADVERTISEMENT' occurrences
    - Decodes HTML entities and strips residual HTML tags
    - Collapses multiple whitespace characters and strips trailing spaces
    - Discards empty strings and standalone scraped tokens (e.g. 'em', 'sup')
    """
    if not isinstance(raw_ingredients, list):
        return []
    
    cleaned_list = []
    for item in raw_ingredients:
        if not isinstance(item, str):
            continue
        # Decode HTML and remove tags
        text = html.unescape(item)
        text = HTML_TAG_PATTERN.sub('', text)
        # Strip advertisement noise
        text = AD_PATTERN.sub('', text)
        # Normalize internal and external whitespace
        text = WHITESPACE_PATTERN.sub(' ', text).strip()
        
        # Filter out empty or noise tokens
        if text and text.lower() not in JUNK_INGREDIENTS and len(text) > 1:
            cleaned_list.append(text)
            
    return cleaned_list

def clean_instructions(raw_instructions: str) -> str:
    """
    Cleans and formats recipe instructions into standardized multi-step text.
    - Decodes HTML entities
    - Converts HTML break tags (<br>, <BR>) into clean line breaks
    - Strips any remaining HTML tags
    - Normalizes line endings and trims whitespace line-by-line
    - Collapses multiple consecutive empty lines
    """
    if pd.isna(raw_instructions) or not isinstance(raw_instructions, str):
        return ""
    
    text = html.unescape(raw_instructions)
    # Replace line break tags with newlines
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = HTML_TAG_PATTERN.sub('', text)
    
    # Process line-by-line: trim lines and remove blank lines
    lines = [WHITESPACE_PATTERN.sub(' ', line).strip() for line in text.splitlines()]
    cleaned_lines = [l for l in lines if l]
    
    return '\n'.join(cleaned_lines)

print("Cleaning helper functions defined successfully.")


Cleaning helper functions defined successfully.


In [7]:
# -----------------------------------------------------------------------------
# STEP 3.2: Applying Cleaning & Extracting NLP Features
# -----------------------------------------------------------------------------
# Apply our modular cleaning functions to df_raw and compute rich metadata features:
# - num_ingredients: count of clean ingredients
# - instruction_len: total character length of instructions
# - num_steps: number of distinct instructional steps/lines

df_clean = df_raw.copy()

print("Applying cleaning functions...")
df_clean["title"] = df_clean["title"].apply(clean_title)
df_clean["ingredients"] = df_clean["ingredients"].apply(clean_ingredients)
df_clean["instructions"] = df_clean["instructions"].apply(clean_instructions)

# Feature Engineering
df_clean["num_ingredients"] = df_clean["ingredients"].apply(len)
df_clean["instruction_len"] = df_clean["instructions"].str.len()
df_clean["num_steps"] = df_clean["instructions"].apply(lambda x: len(x.splitlines()) if x else 0)

# Standardize picture_link: set null or whitespace-only links to None
df_clean["picture_link"] = df_clean["picture_link"].apply(
    lambda x: x.strip() if isinstance(x, str) and x.strip() else None
)

print("Cleaning applied successfully!")
print(f"Sample cleaned recipe (Row 0):\nTitle: {df_clean.loc[0, 'title']}\nIngredients: {df_clean.loc[0, 'ingredients'][:3]}\nNumber of steps: {df_clean.loc[0, 'num_steps']}")


Applying cleaning functions...

Cleaning applied successfully!
Sample cleaned recipe (Row 0):
Title: Slow Cooker Chicken and Dumplings
Ingredients: ['4 skinless, boneless chicken breast halves', '2 tablespoons butter', '2 (10.75 ounce) cans condensed cream of chicken soup']
Number of steps: 2


In [8]:
# -----------------------------------------------------------------------------
# STEP 3.3: Filtering Incomplete & Corrupted Recipes
# -----------------------------------------------------------------------------
# A usable recipe in a cooking or RAG system must fulfill three essential quality rules:
# Rule 1: Non-null, valid title
# Rule 2: At least 1 valid ingredient (num_ingredients > 0)
# Rule 3: Valid instructions (instruction_len >= 10 and contains alphabetic text)

mask_valid_title = df_clean["title"].notna()
mask_valid_ing = df_clean["num_ingredients"] > 0
mask_valid_inst = (df_clean["instruction_len"] >= 10) & df_clean["instructions"].str.contains(r"[a-zA-Z]", regex=True)

valid_mask = mask_valid_title & mask_valid_ing & mask_valid_inst

print("=== Recipe Quality Filter Summary ===")
print(f"Total initial records:      {len(df_clean):,}")
print(f"Invalid / missing title:    {(~mask_valid_title).sum():,}")
print(f"Empty ingredients list:     {(~mask_valid_ing).sum():,}")
print(f"Invalid/short instructions: {(~mask_valid_inst).sum():,}")
print(f"Total invalid rows dropped: {(~valid_mask).sum():,}")
print(f"Remaining high-quality rows: {valid_mask.sum():,} ({(valid_mask.mean() * 100):.2f}%)")

# Keep only valid records
df_clean = df_clean[valid_mask].copy()


=== Recipe Quality Filter Summary ===
Total initial records:      125,164
Invalid / missing title:    575
Empty ingredients list:     2,198
Invalid/short instructions: 734
Total invalid rows dropped: 2,248
Remaining high-quality rows: 122,916 (98.20%)


In [9]:
# -----------------------------------------------------------------------------
# STEP 3.4: Deduplication with Smart Preference
# -----------------------------------------------------------------------------
# Across multiple sources, the same recipe may appear more than once.
# When deduplicating recipes with identical (title, instructions), we prefer
# keeping the record that has an associated image (picture_link is not null).

# Sort so records with valid picture_link come first
df_clean["has_pic"] = df_clean["picture_link"].notna()
df_clean = df_clean.sort_values(by=["has_pic"], ascending=False)

# Drop duplicate recipes based on normalized title and instructions
initial_len = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["title", "instructions"], keep="first")
duplicates_removed = initial_len - len(df_clean)

# Clean up temporary helper column and reset index
df_clean = df_clean.drop(columns=["has_pic"]).reset_index(drop=True)

print(f"Duplicate recipes removed: {duplicates_removed:,}")
print(f"Final clean dataset record count: {len(df_clean):,}")


Duplicate recipes removed: 1,232
Final clean dataset record count: 121,684


In [10]:
# -----------------------------------------------------------------------------
# STEP 4: Post-Cleaning Validation & Quality Comparison
# -----------------------------------------------------------------------------
# Let's verify that all quality issues have been resolved by comparing
# metrics before and after cleaning.

# Check advertisement counts in clean dataset
clean_ad_count = sum(
    1 for ing_list in df_clean["ingredients"]
    for item in ing_list
    if "ADVERTISEMENT" in item.upper()
)

comparison_metrics = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Missing Titles",
        "Empty/Null Ingredients",
        "Empty/Null Instructions",
        "Records with 'ADVERTISEMENT'",
        "Duplicate Recipes"
    ],
    "Raw Dataset": [
        len(df_raw),
        df_raw["title"].isna().sum(),
        df_raw["ingredients"].isna().sum() + df_raw["ingredients"].apply(lambda x: isinstance(x, list) and len(x) == 0).sum(),
        df_raw["instructions"].isna().sum() + (df_raw["instructions"].fillna("").str.strip() == "").sum(),
        standalone_ad_count + appended_ad_count,
        dup_recipes
    ],
    "Cleaned Dataset": [
        len(df_clean),
        df_clean["title"].isna().sum(),
        (df_clean["num_ingredients"] == 0).sum(),
        (df_clean["instruction_len"] == 0).sum(),
        clean_ad_count,
        df_clean.duplicated(subset=["title", "instructions"]).sum()
    ]
})

print("=== Before vs. After Cleaning Comparison ===")
print(comparison_metrics.to_string(index=False))

print("\n=== Cleaned Dataset Summary Statistics ===")
print(df_clean[["num_ingredients", "instruction_len", "num_steps"]].describe())


=== Before vs. After Cleaning Comparison ===
                      Metric  Raw Dataset  Cleaned Dataset
               Total Records       125164           121684
              Missing Titles          569                0
      Empty/Null Ingredients         2193                0
     Empty/Null Instructions         1405                0
Records with 'ADVERTISEMENT'       400811                0
           Duplicate Recipes         1856                0

=== Cleaned Dataset Summary Statistics ===
       num_ingredients  instruction_len      num_steps
count    121684.000000    121684.000000  121684.000000
mean         10.391235       997.715460       4.027514
std           5.360729       886.190404       2.637029
min           1.000000        11.000000       1.000000
25%           7.000000       453.000000       2.000000
50%           9.000000       751.000000       4.000000
75%          13.000000      1246.000000       5.000000
max          82.000000     28058.000000      71.000000


In [11]:
# -----------------------------------------------------------------------------
# STEP 5: Exporting Cleaned Dataset for Downstream Usage
# -----------------------------------------------------------------------------
# We save the cleaned DataFrame to 'data_recipes/recipes_cleaned.json'.
# This file is now fully prepared for vector store ingestion, chunking, and search.

output_file = "data_recipes/recipes_cleaned.json"

# Select and order the final columns
export_cols = [
    "id",
    "title",
    "ingredients",
    "instructions",
    "picture_link",
    "source",
    "num_ingredients",
    "num_steps",
    "instruction_len"
]
df_export = df_clean[export_cols]

# Save as formatted JSON records
print(f"Exporting {len(df_export):,} cleaned recipes to '{output_file}'...")
df_export.to_json(output_file, orient="records", indent=2)

file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"Export complete! File size: {file_size_mb:.2f} MB")

# Verify the file can be read back cleanly
test_df = pd.read_json(output_file)
print(f"Verification: Successfully reloaded {len(test_df):,} records from JSON.")
print("\nFirst cleaned record preview:")
print(f"Title: {test_df.iloc[0]['title']}")
print(f"Ingredients: {test_df.iloc[0]['ingredients']}")
print(f"Instructions:\n{test_df.iloc[0]['instructions']}")


Exporting 121,684 cleaned recipes to 'data_recipes/recipes_cleaned.json'...


Export complete! File size: 199.46 MB


Verification: Successfully reloaded 121,684 records from JSON.

First cleaned record preview:
Title: Slow Cooker Chicken and Dumplings
Ingredients: ['4 skinless, boneless chicken breast halves', '2 tablespoons butter', '2 (10.75 ounce) cans condensed cream of chicken soup', '1 onion, finely diced', '2 (10 ounce) packages refrigerated biscuit dough, torn into pieces']
Instructions:
Place the chicken, butter, soup, and onion in a slow cooker, and fill with enough water to cover.
Cover, and cook for 5 to 6 hours on High. About 30 minutes before serving, place the torn biscuit dough in the slow cooker. Cook until the dough is no longer raw in the center.
